# Level3 JAX PPO Training Notebook

这个 notebook 复用现有 Level3 PPO notebook 的训练流程，但 rollout 走 `env._step + jax.lax.scan`，actor、critic、GAE、PPO update、checkpoint 和 eval 都走 `lsy_drone_racing.control.jax_ppo_fast` 的 JAX/Optax fast path。

推荐顺序：

1. 检查 kernel 和仓库路径。
2. 跑零动作 debug rollout 看 observation / reward。
3. 跑 JAX PPO smoke train，确认 rollout、GAE、Optax update 和 checkpoint 保存正常。
4. 用 Level2 Fast2 PPO 参数跑 JAX 架构验证。
5. 调大 `num_envs` / `total_timesteps` 后正式训练 Level3 JAX PPO。


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.55")

ROOT = Path.cwd().resolve()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("repo:", ROOT)
print("python:", sys.executable)
print("XLA preallocate:", os.environ.get("XLA_PYTHON_CLIENT_PREALLOCATE"))
print("XLA memory fraction:", os.environ.get("XLA_PYTHON_CLIENT_MEM_FRACTION"))


如果 import 失败，通常是 kernel 没选对。新增 JAX PPO 核心使用 JAX/Optax，但为了完全复用当前项目的环境 wrapper，仍需要能 import 现有 CleanRL 文件所在环境。

正式 GPU 训练请选择：

```text
.pixi/envs/gpu/bin/python
```

CPU debug 可以选择：

```text
.pixi/envs/tests/bin/python
```

如果 VS Code 不显示 kernel，在项目根目录运行：

```bash
pixi run -e gpu python -m ipykernel install --user --name lsy-drone-racing-gpu --display-name "LSY Drone Racing (gpu)"
pixi run -e tests python -m ipykernel install --user --name lsy-drone-racing-tests --display-name "LSY Drone Racing (tests)"
```


In [ ]:
%reload_ext autoreload
%autoreload 2

import json
import numpy as np
import jax
import optax

from lsy_drone_racing.control import jax_ppo, jax_ppo_fast
from lsy_drone_racing.control.jax_ppo import (
    JaxPPOArgs,
    load_jax_checkpoint,
)
from lsy_drone_racing.control.jax_ppo_fast import (
    benchmark_fast_rollout,
    device_put_tree,
    evaluate_ppo_fast,
    level2_validation_args,
    make_fast_base_env,
    make_initial_state,
    train_ppo_fast,
)

print("jax:", jax.__version__)
print("devices:", jax.devices())
print("optax:", optax.__version__)
print("checkpoint format:", jax_ppo.CHECKPOINT_FORMAT)


## 调参


In [ ]:
TRAIN_ENV_TOML = "level3_dr.toml"
EVAL_ENV_TOML = TRAIN_ENV_TOML
RUN_NAME = "jax_level3_ppo"
CONFIG = TRAIN_ENV_TOML

CONFIG_PATH = ROOT / "config" / TRAIN_ENV_TOML
if not CONFIG_PATH.exists():
    raise FileNotFoundError(CONFIG_PATH)
EVAL_CONFIG_PATH = ROOT / "config" / EVAL_ENV_TOML
if not EVAL_CONFIG_PATH.exists():
    raise FileNotFoundError(EVAL_CONFIG_PATH)

CHECKPOINT_DIR = ROOT / f"lsy_drone_racing/control/checkpoints/{RUN_NAME}"
MODEL_PATH = CHECKPOINT_DIR / f"{RUN_NAME}_final.pkl"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

JAX_DEVICE = "gpu"
DEBUG_JAX_DEVICE = "cpu"

WANDB_ENABLED = True
WANDB_PROJECT_NAME = "ADR-PPO-Racing"
WANDB_ENTITY = None
WANDB_MODE = "online"

NUM_ENVS_DEBUG = 4
NUM_ENVS_TRAIN = 1024
NUM_STEPS = 32
TOTAL_TIMESTEPS_TRAIN = 300_000_000
CHECKPOINT_INTERVAL_STEPS = 10_000_000
LEARNING_RATE = 3e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95
UPDATE_EPOCHS = 5
NUM_MINIBATCHES = 8
ENT_COEF = 0.02
TARGET_KL = 0.03
HIDDEN_DIM = 256

REWARD_KWARGS = dict(
    progress_coef=0.0,
    gate_stage_coef=15.0,
    gate_axis_coef=15.0,
    near_gate_coef=0.0,
    gate_bonus=120.0,
    gate_back_bonus=20.0,
    finish_bonus=160.0,
    missed_gate_penalty=0.0,
    wrong_side_penalty=6.0,
    crash_penalty=50.0,
    obstacle_coef=5.0,
    obstacle_margin=0.3,
    obstacle_clearance_coef=0.0,
    timeout_penalty=80.0,
    time_penalty=0.03,
    act_coef=0.03,
    d_act_th_coef=0.10,
    d_act_xy_coef=0.10,
    cmd_tilt_coef=1.0,
    rpy_coef=1.0,
    tilt_limit_deg=40.0,
    tilt_excess_coef=15.0,
)

def build_level3_args(**overrides):
    base = dict(
        level="level3",
        config=CONFIG,
        jax_device=JAX_DEVICE,
        wandb_project_name=WANDB_PROJECT_NAME,
        wandb_entity=WANDB_ENTITY,
        wandb_mode=WANDB_MODE,
        num_steps=NUM_STEPS,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA,
        update_epochs=UPDATE_EPOCHS,
        num_minibatches=NUM_MINIBATCHES,
        learning_rate=LEARNING_RATE,
        ent_coef=ENT_COEF,
        target_kl=TARGET_KL,
        hidden_dim=HIDDEN_DIM,
        n_obs=2,
        **REWARD_KWARGS,
    )
    base.update(overrides)
    return JaxPPOArgs.create(**base)

MODEL_PATH


## 1. 检查 observation / reward

这一格不会训练，只用零动作跑几步。重点看 observation shape 是否稳定、reward 分量是否有非有限值，以及是否一开始大量终止。


In [ ]:
debug_args = build_level3_args(
    num_envs=NUM_ENVS_DEBUG,
    total_timesteps=NUM_ENVS_DEBUG * NUM_STEPS,
    jax_device=DEBUG_JAX_DEVICE,
    debug_reward_every=1,
)

benchmark_fast_rollout(debug_args, repeat=1, warmup=0)


## 2. 手动创建 JAX 环境看 shape


In [ ]:
shape_args = build_level3_args(
    num_envs=NUM_ENVS_DEBUG,
    jax_device=DEBUG_JAX_DEVICE,
)
envs, cfg, action_low, action_high = make_fast_base_env(
    level=shape_args.level,
    config=shape_args.config,
    num_envs=shape_args.num_envs,
    jax_device=shape_args.jax_device,
)
raw_obs, info = envs.reset(seed=shape_args.seed)
raw_obs = device_put_tree(raw_obs, jax.devices(shape_args.jax_device)[0])
state = make_initial_state(envs, raw_obs, shape_args, action_low, action_high)
print("fast obs shape:", tuple(np.asarray(state.obs).shape))
print("single action space:", envs.single_action_space)
print("sim freq:", cfg.sim.freq, "policy freq:", cfg.env.freq)
envs.close()


## 3. JAX PPO smoke train

这个只训练几十步，目标是确认 fast JAX rollout、GAE、Optax update、checkpoint 保存和加载都不报错。


In [ ]:
smoke_args = build_level3_args(
    num_envs=2,
    num_steps=8,
    num_minibatches=2,
    total_timesteps=64,
    jax_device="cpu",
)

smoke_path = Path("/tmp/jax_ppo_level3_notebook_smoke.pkl")
train_ppo_fast(
    smoke_args,
    smoke_path,
    wandb_enabled=False,
)
load_jax_checkpoint(smoke_path)


## 4. Level2 Fast2 参数验证 JAX 架构

完整验证目标是：用 Level2 Fast2 的 PPO 参数训练 fast JAX checkpoint，并在 100 个 Level2 seed 上复现高成功率。当前 fast path 实测 `success_rate = 0.79`，你已确认 79% 可接受。下面先给 smoke 版本；正式验证请运行后一格终端命令。


In [ ]:
level2_smoke_args = level2_validation_args(
    jax_device="cpu",
    num_envs=4,
    num_steps=8,
    num_minibatches=2,
    total_timesteps=64,
)
level2_smoke_path = Path("/tmp/jax_ppo_level2_fast2_smoke.pkl")
train_ppo_fast(level2_smoke_args, level2_smoke_path, wandb_enabled=False)
evaluate_ppo_fast(level2_smoke_args, n_eval=2, model_path=level2_smoke_path, seed_start=1)


In [ ]:
# 正式验证会跑完整 Level2 Fast2 参数，耗时较长。
# 需要时在终端执行：
print(
    "pixi run -e gpu python scripts/validate_level2_jax_ppo_fast.py "
    "--eval-episodes 100 --eval-seed-start 1 --success-threshold 0.79"
)


## 5. 正式训练 Level3 JAX PPO


In [ ]:
train_args = build_level3_args(
    num_envs=NUM_ENVS_TRAIN,
    num_steps=NUM_STEPS,
    total_timesteps=TOTAL_TIMESTEPS_TRAIN,
    wandb_run_name=RUN_NAME,
    wandb_run_id=RUN_NAME,
)

train_ppo_fast(
    train_args,
    MODEL_PATH,
    wandb_enabled=WANDB_ENABLED,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_interval=CHECKPOINT_INTERVAL_STEPS,
)
print("saved:", MODEL_PATH)


## 6. 快速评估 Level3 JAX checkpoint

这里评估的是训练 wrapper 下的 deterministic policy success/reward/length，不等于最终比赛部署 controller 评估。


In [ ]:
eval_args = build_level3_args(
    config=EVAL_ENV_TOML,
    num_envs=1,
    jax_device="cpu",
)
summary = evaluate_ppo_fast(eval_args, n_eval=5, model_path=MODEL_PATH, seed_start=eval_args.seed + 1)
print(json.dumps({k: v for k, v in summary.items() if k != "episodes_detail"}, indent=2))
